In [2]:
# ======================================================
# Step 1: 环境设置与依赖安装
# ======================================================

%pip install transformers datasets tokenizers jinja2

# 导入所有需要的库
import random
import json
import os
from transformers import AutoTokenizer, PreTrainedTokenizerFast
from tokenizers import (
    decoders,
    models,
    pre_tokenizers,
    trainers,
    Tokenizer,
)
from tokenizers.normalizers import NFKC
from typing import Generator

print("所有依赖已安装并导入成功！")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
e:\资料\课程资料\大三上\happy-llm\my_experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


所有依赖已安装并导入成功！


In [3]:
# ======================================================
# Step 2: 准备并加载训练数据
# ======================================================

import os
import json
from typing import Generator

# --- 1. 安装 huggingface-cli 的依赖 ---
%pip install huggingface_hub

# --- 2. 设置镜像加速下载 ---
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# --- 3. 下载 BelleGroup/train_0.5M_CN 数据集 ---
# --repo-type dataset: 表明要下载的是一个数据集
# --resume-download: 支持断点续传
# BelleGroup/train_0.5M_CN: 数据集ID
# --local-dir: 指定下载到本地的哪个文件夹
dataset_dir = "./belle_data_0.5M"
print(f"开始下载 BelleGroup/train_0.5M_CN 数据集到 '{dataset_dir}' 文件夹...")
!huggingface-cli download --repo-type dataset --resume-download BelleGroup/train_0.5M_CN --local-dir {dataset_dir}

print("\n数据集下载完成！")


# --- 4. 定义数据文件路径并进行验证 ---
# 下载后的文件名是 Belle_open_source_0.5M.json
data_path = os.path.join(dataset_dir, "Belle_open_source_0.5M.json")

# 验证文件是否存在，并查看文件大小
try:
    file_size = os.path.getsize(data_path) / (1024 * 1024) # 转换为 MB
    print(f"成功找到数据文件: '{data_path}'")
    print(f"文件大小: {file_size:.2f} MB")
except FileNotFoundError:
    print(f"错误：未找到数据文件 '{data_path}'。请检查下载步骤是否出错。")
    raise Exception("数据文件缺失，无法继续。") from None

# --- 5. 定义数据读取函数 (指令格式) ---
def read_texts_from_json(file_path: str) -> Generator[str, None, None]:
    """
    读取指令格式的json文件，并拼接所有文本字段。
    """
    print(f"\n开始从 '{file_path}' 以指令格式读取数据...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            try:
                data = json.loads(line)
                
                # 从 instruction, input, output 字段提取文本
                instruction = data.get("instruction", "")
                inp = data.get("input", "")
                output = data.get("output", "")
                
                # 将所有文本字段拼接成一个长字符串，用换行符分隔
                full_text = f"{instruction}{inp}{output}"
                
                # 移除多余的换行符和空白符并返回
                yield full_text.replace('\n', ' ').strip()

            except json.jsonDecodeError:
                if line_num <= 5:
                    print(f"警告: 第 {line_num} 行 json 解析错误，已跳过。")
                continue

# --- 6. 测试读取函数 ---
print("\n测试读取前3条数据:")
count = 0
for text_sample in read_texts_from_json(data_path):
    print(f"--- 样本 {count+1} ---")
    # 只打印前150个字符，避免过长
    print(text_sample[:150].replace('\n', ' \\n ') + "...")
    count += 1
    if count >= 3:
        break

Note: you may need to restart the kernel to use updated packages.
开始下载 BelleGroup/train_0.5M_CN 数据集到 './belle_data_0.5M' 文件夹...



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
Returning existing local_dir `belle_data_0.5M` as remote repo cannot be accessed in `snapshot_download` ((ProtocolError('Connection aborted.', ConnectionAbortedError(10053, '你的主机中的软件中止了一个已建立的连接。', None, 10053, None)), '(Request ID: 3d692d05-6efb-4d7d-9909-cdfecec5b4f9)')).
Exception ignored in: <generator object read_texts_from_json at 0x0000021D2FA27810>
Traceback (most recent call last):
  File "C:\Users\71808\AppData\Local\Temp\ipykernel_82016\2706402157.py", line 62, in read_texts_from_json
AttributeError: module 'json' has no attribute 'jsonDecodeError'


⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
E:\资料\课程资料\大三上\happy-llm\my_experiments\1_tokenizer_training\belle_data_0.5M

数据集下载完成！
成功找到数据文件: './belle_data_0.5M\Belle_open_source_0.5M.json'
文件大小: 272.77 MB

测试读取前3条数据:

开始从 './belle_data_0.5M\Belle_open_source_0.5M.json' 以指令格式读取数据...
--- 样本 1 ---
给定一个英文句子，翻译成中文。 I love to learn new things every day. 我每天喜欢学习新事物。...
--- 样本 2 ---
给定一个文字输入，将其中的所有数字加1。 “明天的会议在9点开始，记得准时到达。” “明天的会议在10点开始，记得准时到达。”...
--- 样本 3 ---
根据以下信息创建一个新的电子邮件账户：您的用户名应该包含您的姓氏和您的生日，在.com域中注册一个电子邮件地址，并将生成的用户名和密码作为输出提供。 姓氏：李 生日：1990年1月1日 生成的用户名应该是li19900101，并在.com域中注册电子邮件地址。由于安全和隐私原因，我无法提供您所需的密...


In [4]:
# ======================================================
# Step 3: 创建配置文件生成函数
# ======================================================

def create_tokenizer_config(save_dir: str) -> None:
    """创建完整的tokenizer配置文件"""
    config = {
        "add_bos_token": False,
        "add_eos_token": False,
        "add_prefix_space": False,
        "bos_token": "<|im_start|>",
        "eos_token": "<|im_end|>",
        "pad_token": "<|im_end|>",
        "unk_token": "<unk>",
        "model_max_length": 1000000000000000019884624838656,
        "clean_up_tokenization_spaces": False,
        "tokenizer_class": "PreTrainedTokenizerFast",
        "chat_template": (
            "{% for message in messages %}"
            "{% if message['role'] == 'system' %}"
            "<|im_start|>system\n{{ message['content'] }}<|im_end|>\n"
            "{% elif message['role'] == 'user' %}"
            "<|im_start|>user\n{{ message['content'] }}<|im_end|>\n"
            "{% elif message['role'] == 'assistant' %}"
            "<|im_start|>assistant\n{{ message['content'] }}<|im_end|>\n"
            "{% endif %}"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "{{ '<|im_start|>assistant\n' }}"
            "{% endif %}"
        )
    }

    # 保存主配置文件
    with open(os.path.join(save_dir, "tokenizer_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=4)

    # 创建special_tokens_map.json
    special_tokens_map = {
        "bos_token": "<|im_start|>",
        "eos_token": "<|im_end|>",
        "unk_token": "<unk>",
        "pad_token": "<|im_end|>",
        "additional_special_tokens": ["<s>", "</s>"]
    }
    with open(os.path.join(save_dir, "special_tokens_map.json"), "w", encoding="utf-8") as f:
        json.dump(special_tokens_map, f, ensure_ascii=False, indent=4)

print("配置文件生成函数定义成功！")

配置文件生成函数定义成功！


In [5]:
# ======================================================
# Step 4: 训练 BPE Tokenizer
# ======================================================

def train_tokenizer(data_path: str, save_dir: str, vocab_size: int = 8192) -> None:
    """训练并保存自定义tokenizer"""
    os.makedirs(save_dir, exist_ok=True)
    
    # 初始化tokenizer
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = NFKC()  # 添加文本规范化
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()

    # 配置特殊token
    special_tokens = [
        "<unk>", 
        "<s>", 
        "</s>", 
        "<|im_start|>", 
        "<|im_end|>"
    ]

    # 配置训练器
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=special_tokens,
        min_frequency=2,  # 提高低频词过滤
        show_progress=True,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
    )

    # 训练tokenizer
    print(f"Training tokenizer with data from {data_path}")
    texts = read_texts_from_json(data_path)
    tokenizer.train_from_iterator(texts, trainer=trainer, length=os.path.getsize(data_path))

    # 验证特殊token映射
    try:
        assert tokenizer.token_to_id("<unk>") == 0
        assert tokenizer.token_to_id("<s>") == 1
        assert tokenizer.token_to_id("</s>") == 2
        assert tokenizer.token_to_id("<|im_start|>") == 3
        assert tokenizer.token_to_id("<|im_end|>") == 4
    except AssertionError as e:
        print("Special tokens mapping error:", e)
        raise

    # 保存tokenizer文件
    tokenizer.save(os.path.join(save_dir, "tokenizer.json"))
    
    # 创建配置文件
    create_tokenizer_config(save_dir)
    print(f"Tokenizer saved to {save_dir}")

# --- 开始训练 ---
tokenizer_save_dir = "my_tokenizer"
# 使用我们创建的微型数据集进行训练
train_tokenizer(data_path=data_path, save_dir=tokenizer_save_dir)

print("\n训练完成！")

Training tokenizer with data from ./belle_data_0.5M\Belle_open_source_0.5M.json

开始从 './belle_data_0.5M\Belle_open_source_0.5M.json' 以指令格式读取数据...
Tokenizer saved to my_tokenizer

训练完成！


In [9]:
# ======================================================
# Step 5: 使用和评估训练好的 Tokenizer
# ======================================================

def eval_tokenizer(tokenizer_path: str) -> None:
    """评估tokenizer功能"""
    try:
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    except Exception as e:
        print(f"Error loading tokenizer: {e}")
        return

    # 测试基本属性
    print("\n=== Tokenizer基本信息 ===")
    print(f"Vocab size: {len(tokenizer)}")
    print(f"Special tokens: {tokenizer.all_special_tokens}")
    print(f"Special token IDs: {tokenizer.all_special_ids}")

    # 测试聊天模板
    messages = [
        {"role": "system", "content": "你是一个AI助手。"},
        {"role": "user", "content": "How are you?"},
        {"role": "assistant", "content": "I'm fine, thank you. and you?"},
        {"role": "user", "content": "I'm good too."},
        {"role": "assistant", "content": "That's great to hear!"},
    ]
    
    print("\n=== 聊天模板测试 ===")
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        # add_generation_prompt=True
    )
    print("Generated prompt:\n", prompt, sep="")

    # 测试编码解码
    print("\n=== 编码解码测试 ===")
    # 编码: 将字符串 prompt 转换为数字 ID 列表
    encoded = tokenizer(prompt, truncation=True, max_length=256)
    
    # 获取编码后的 input_ids
    input_ids = encoded["input_ids"]
    
    print(f"\n原始长字符串 prompt (部分): \n'{prompt[:50]}...'")
    print(f"\n编码后的 input_ids (部分): \n{input_ids[:20]}...")
    print(f"编码后的序列总长度: {len(input_ids)}")

    print("\n手动解码前几个 ID:")
    print(f"ID 3  -> '{tokenizer.decode([3])}'")
    print(f"ID {input_ids[1]} -> '{tokenizer.decode([input_ids[1]])}'")
    print(f"ID {input_ids[2]} -> '{tokenizer.decode([input_ids[2]])}'")
    print(f"ID {input_ids[3]} -> '{tokenizer.decode([input_ids[3]])}'")
    
    # 解码: 将数字 ID 列表转换回字符串
    decoded = tokenizer.decode(input_ids, skip_special_tokens=False)
    print("\nDecoded text matches original:", decoded == prompt)

    # 测试特殊token处理
    print("\n=== 特殊token处理 ===")
    test_text = "<|im_start|>user\nHello<|im_end|>"
    encoded = tokenizer(test_text).input_ids
    decoded = tokenizer.decode(encoded)
    print(f"Original: {test_text}")
    print(f"Decoded:  {decoded}")
    print("Special tokens preserved:", decoded == test_text)

# --- 开始评估 ---
eval_tokenizer(tokenizer_save_dir)


=== Tokenizer基本信息 ===
Vocab size: 8192
Special tokens: ['<|im_start|>', '<|im_end|>', '<unk>', '<s>', '</s>']
Special token IDs: [3, 4, 0, 1, 2]

=== 聊天模板测试 ===
Generated prompt:
<|im_start|>system
你是一个AI助手。<|im_end|>
<|im_start|>user
How are you?<|im_end|>
<|im_start|>assistant
I'm fine, thank you. and you?<|im_end|>
<|im_start|>user
I'm good too.<|im_end|>
<|im_start|>assistant
That's great to hear!<|im_end|>


=== 编码解码测试 ===

原始长字符串 prompt (部分): 
'<|im_start|>system
你是一个AI助手。<|im_end|>
<|im_start|...'

编码后的 input_ids (部分): 
[3, 87, 93, 7965, 203, 384, 975, 1344, 3538, 265, 4, 203, 3, 1410, 497, 203, 44, 1358, 2275, 1443]...
编码后的序列总长度: 69

手动解码前几个 ID:
ID 3  -> '<|im_start|>'
ID 87 -> 's'
ID 93 -> 'y'
ID 7965 -> 'stem'

Decoded text matches original: True

=== 特殊token处理 ===
Original: <|im_start|>user
Hello<|im_end|>
Decoded:  <|im_start|>user
Hello<|im_end|>
Special tokens preserved: True
